In [49]:
#This code compresses PROCEDURES_ICD9, and DIAGNOSES_ICD9 into single entries per admission (sorted by SEQ_NUM)
#and merges this data with the ADMISSIONS dataframe, providing two separate dataframes with this merged structure.
#One dataframe corresponds to patients diagnosed with specified ICD-9 codes and the other contains the control patients.

import numpy as np
import pandas as pd
import seaborn as sns
import datetime
import matplotlib.pyplot as plt

ADMISSIONS = pd.read_csv("./ADMISSIONS.csv")
DIAGNOSES_ICD = pd.read_csv("./DIAGNOSES_ICD.csv")
PROCEDURES_ICD = pd.read_csv("./PROCEDURES_ICD.csv")

In [50]:
# Count rare diagnoses codes for filtering, to reduce low frequency features
diagnosis_counts = DIAGNOSES_ICD['ICD9_CODE'].value_counts()
rare_diags = (diagnosis_counts < 100).sum()

# Count rare Procedures for the same purpose
procedure_counts = PROCEDURES_ICD['ICD9_CODE'].value_counts()
rare_procs = (procedure_counts < 100).sum()

print(f"Diagnoses appearing < 100 times: {rare_diags}")
print(f"Procedures appearing < 100 times: {rare_procs}")

# Identify codes appearing 100+ times
common_procedures = procedure_counts[procedure_counts >= 100].index
common_diagnoses = diagnosis_counts[diagnosis_counts >= 100].index

#Input ICD9 code for our analysis here
my_icd9_code = ["4210"]

# Only extract diagnoses and procedures appearing greater than 100 times. 
DIAGNOSES_ICD = DIAGNOSES_ICD[DIAGNOSES_ICD['ICD9_CODE'].isin(common_diagnoses) | DIAGNOSES_ICD['ICD9_CODE'].isin(my_icd9_code)]
PROCEDURES_ICD = PROCEDURES_ICD[PROCEDURES_ICD['ICD9_CODE'].isin(common_procedures) | PROCEDURES_ICD['ICD9_CODE'].isin(my_icd9_code)]

print(f"Final filtered diagnoses count: {DIAGNOSES_ICD['ICD9_CODE'].nunique()}")
print(f"Final filtered procedures count: {PROCEDURES_ICD['ICD9_CODE'].nunique()}")

#Returns patients with target disease
DISEASE_SUBJECT_ID = DIAGNOSES_ICD.loc[
    DIAGNOSES_ICD["ICD9_CODE"].astype(str).isin(my_icd9_code),
    "SUBJECT_ID"
].unique()

# Also grab patients with target disease in admission text
disease_text_ids = ADMISSIONS[
    ADMISSIONS['DIAGNOSIS'].str.contains('BACTERIAL ENDOCARDITIS', na=False)
]['SUBJECT_ID'].unique()

# Combine both sets — these should ALL be excluded from controls
EXCLUDE_FROM_CONTROLS = np.union1d(DISEASE_SUBJECT_ID, disease_text_ids)

# Now build the entire control group dataset excluding patients with target disease either in text or ICD-9 codes
CONTROL_SUBJECT_ID = DIAGNOSES_ICD.loc[
    ~DIAGNOSES_ICD["SUBJECT_ID"].isin(EXCLUDE_FROM_CONTROLS),
    "SUBJECT_ID"
].unique()

#Returns the specific admissions where target disease was diagnosed
DISEASE_HADM_ID = DIAGNOSES_ICD.loc[
    DIAGNOSES_ICD["ICD9_CODE"].astype(str).isin(my_icd9_code),
    "HADM_ID"
].unique()

#Identify all diagnoses for patients diagnosed with target disease, including for admissions where they were not diagnosed with disease
PATIENT_DIAGNOSES = DIAGNOSES_ICD[DIAGNOSES_ICD['SUBJECT_ID'].isin(DISEASE_SUBJECT_ID)]

#Identify all diagnoses for control patients as well, excluding any with the target disease
CONTROL_DIAGNOSES = DIAGNOSES_ICD[DIAGNOSES_ICD['SUBJECT_ID'].isin(CONTROL_SUBJECT_ID)]

#filter out target diagnosis so that it does not cause data leakage in the ML model, while retaining the patients with the disease
PATIENT_DIAGNOSES = PATIENT_DIAGNOSES[
    ~PATIENT_DIAGNOSES['ICD9_CODE'].isin(my_icd9_code)
]

#Return a new dataframe with all the ICD9 codes for each admission condensed into a single row,col val as a compressed list
PATIENT_DIAGNOSES = (
    PATIENT_DIAGNOSES
    .sort_values(['HADM_ID','SEQ_NUM'])
    .groupby(['SUBJECT_ID','HADM_ID'])['ICD9_CODE']
    .apply(list)
    .reset_index(name='DIAGNOSES')
)

#Return a new dataframe with all the ICD9 codes for each admission condensed into a single row,col val as a compressed list for control patients
CONTROL_DIAGNOSES = (
    CONTROL_DIAGNOSES
    .sort_values(['HADM_ID','SEQ_NUM'])
    .groupby(['SUBJECT_ID','HADM_ID'])['ICD9_CODE']
    .apply(list)
    .reset_index(name='DIAGNOSES')
)

#Remove DIAGNOSES_ICD to conserve memory since we have already filtered for the relevant data
#del DIAGNOSES_ICD

#Return all procedures for patients diagnosed with AD, including for admissions where they were not diagnosed with AD
PATIENT_PROCEDURES = PROCEDURES_ICD[PROCEDURES_ICD['SUBJECT_ID'].isin(DISEASE_SUBJECT_ID)]

#identify all procedures for control patients as well
CONTROL_PROCEDURES = PROCEDURES_ICD[PROCEDURES_ICD['SUBJECT_ID'].isin(CONTROL_SUBJECT_ID)]

#Return a new dataframe with all procedure codes for each admission compressed into a single row,col val as a compressed list
PATIENT_PROCEDURES = (
    PATIENT_PROCEDURES
    .sort_values(['HADM_ID','SEQ_NUM'])
    .groupby(['SUBJECT_ID','HADM_ID'])['ICD9_CODE']
    .apply(list)
    .reset_index(name='PROCEDURE TYPE')
)

#Return a new dataframe with all procedure codes for each admission compressed into a single row,col val as a compressed list for control patients
CONTROL_PROCEDURES = (
    CONTROL_PROCEDURES
    .sort_values(['HADM_ID','SEQ_NUM'])
    .groupby(['SUBJECT_ID','HADM_ID'])['ICD9_CODE']
    .apply(list)
    .reset_index(name='PROCEDURE TYPE')
)

#Return every admission entry for patients who were diagnosed with target disease at some point
PATIENT_ADMISSIONS = ADMISSIONS[ADMISSIONS['SUBJECT_ID'].isin(DISEASE_SUBJECT_ID)]

#pull control group admissions as well
CONTROL_ADMISSIONS = ADMISSIONS[ADMISSIONS['SUBJECT_ID'].isin(CONTROL_SUBJECT_ID)]

#Merge the compressed DFs engineered earlier with admissions so that each admission has diagnosis, and procedure data
PATIENT_ADMISSIONS_MERGED = PATIENT_ADMISSIONS.merge(PATIENT_DIAGNOSES, on=["HADM_ID","SUBJECT_ID"], how="left") \
            .merge(PATIENT_PROCEDURES, on=["HADM_ID","SUBJECT_ID"], how="left")

CONTROL_ADMISSIONS_MERGED = CONTROL_ADMISSIONS.merge(CONTROL_DIAGNOSES, on=["HADM_ID","SUBJECT_ID"], how="left") \
            .merge(CONTROL_PROCEDURES, on=["HADM_ID","SUBJECT_ID"], how="left")

#Rename columns for clarity since there is a text-based labeling column and the ICD-9 diagnosis column
PATIENT_ADMISSIONS_MERGED = PATIENT_ADMISSIONS_MERGED.rename(columns={"DIAGNOSIS": "DIAGNOSIS (LABEL)","DIAGNOSES": "DIAGNOSIS (ICD_9)"})
CONTROL_ADMISSIONS_MERGED = CONTROL_ADMISSIONS_MERGED.rename(columns={"DIAGNOSIS": "DIAGNOSIS (LABEL)","DIAGNOSES": "DIAGNOSIS (ICD_9)"})

#Drop redundant row
PATIENT_ADMISSIONS_MERGED.drop(['ROW_ID'],inplace=True,axis=1)
CONTROL_ADMISSIONS_MERGED.drop(['ROW_ID'],inplace=True,axis=1)

#Identify the admissions where target disease was one of the diagnoses given to the patients, excluding admissions where it was not diagnosed
#No need to do this for control group
DISEASE_ADMISSIONS = PATIENT_ADMISSIONS_MERGED[PATIENT_ADMISSIONS_MERGED['HADM_ID'].isin(DISEASE_HADM_ID)]
DISEASE_ADMISSIONS = DISEASE_ADMISSIONS.copy()

#Convert ADMITTIME to datetime for processing
DISEASE_ADMISSIONS['ADMITTIME'] = pd.to_datetime(DISEASE_ADMISSIONS["ADMITTIME"], errors="coerce")
CONTROL_ADMISSIONS_MERGED['ADMITTIME'] = pd.to_datetime(CONTROL_ADMISSIONS_MERGED["ADMITTIME"], errors="coerce")

#Sort by HADM_ID and ADMITTIME to get a sorted list for processing
DISEASE_ADMISSIONS = DISEASE_ADMISSIONS.sort_values(['HADM_ID','ADMITTIME'])
CONTROL_ADMISSIONS_MERGED = CONTROL_ADMISSIONS_MERGED.sort_values(['HADM_ID','ADMITTIME'])

#Identify the earliest admission time in which patients were diagnosed with target disease
DISEASE_FIRST_ADMISSIONS = DISEASE_ADMISSIONS.groupby('SUBJECT_ID',as_index=False)['ADMITTIME'].min()

#Rename this column to "Comparator" since it will be used for filtering admissions from after the patient was diagnosed with target disease
DISEASE_FIRST_ADMISSIONS = DISEASE_FIRST_ADMISSIONS.rename(columns={"ADMITTIME": "Comparator"})

#Update ADMISSIONS_MERGED so it now contains all admissions for patients who were diagnosed with target disease at some point
#Prior and including the admission with their first diagnosis of AD. Admissions after their first diagnosis are excluded
PATIENT_ADMISSIONS_MERGED['ADMITTIME'] = pd.to_datetime(PATIENT_ADMISSIONS_MERGED["ADMITTIME"], errors="coerce")
PATIENT_ADMISSIONS_MERGED = PATIENT_ADMISSIONS_MERGED.merge(DISEASE_FIRST_ADMISSIONS,on='SUBJECT_ID',how="left")
PATIENT_ADMISSIONS_MERGED = PATIENT_ADMISSIONS_MERGED[PATIENT_ADMISSIONS_MERGED['ADMITTIME']<PATIENT_ADMISSIONS_MERGED['Comparator']]

#drop the comparator column now that filtering is done so that the DFs are the same
PATIENT_ADMISSIONS_MERGED = PATIENT_ADMISSIONS_MERGED.drop(['Comparator'],axis=1)

# Ensure data is sorted by patient, with most recent Admission first
PATIENT_ADMISSIONS_MERGED = PATIENT_ADMISSIONS_MERGED.sort_values(['SUBJECT_ID', 'ADMITTIME'], ascending=[True, False])

# Count down starting from most recent admission prior to admission of first diagnosis
PATIENT_ADMISSIONS_MERGED['event_index'] = (PATIENT_ADMISSIONS_MERGED.groupby('SUBJECT_ID').cumcount() + 1) * -1

# Determine the number of controls we want to use (same as number of patients with condition)
num_controls = len(PATIENT_ADMISSIONS_MERGED['SUBJECT_ID'].unique())

# Randomly pick control subjects using np.random.choice, ensuring we have the same number of unique control subjects
sampled_control_ids = np.random.choice(CONTROL_SUBJECT_ID, num_controls, replace=False)

# Filter controls to only include those randomly sampled control subjects
CONTROL_ADMISSIONS_MERGED = CONTROL_ADMISSIONS_MERGED[
    CONTROL_ADMISSIONS_MERGED['SUBJECT_ID'].isin(sampled_control_ids)
].copy()

# Number of admissions captured for each patient prior to admission of diagnosis. We are extracting this so we can impute the same distribution into the control group. 
# We do not want the model to learn based on differences in this column between the two groups. We also want the controls group not to have a different number of procedures and diagnoses columns
# which will be further demonstrated in the next pre processing code snippet. This is a critical step to ensure that the model learns based on the content of the admissions and not based on differences in the number of admissions captured for each group, which would be a form of data leakage.
disease_counts = PATIENT_ADMISSIONS_MERGED.groupby('SUBJECT_ID').size()

# Extract all of the unique SUBJECT_IDs within our new controls DF
unique_controls = CONTROL_ADMISSIONS_MERGED['SUBJECT_ID'].unique()

#Randomly sample from distribution of admission count for each patient in disease group, so we can assign for control group patients. This ensures that the admissions index distribution will not differ between disease and controls.
sampled_depths = disease_counts.sample(len(unique_controls)).values

#Assign a theoretical "admissions index" to each control patient based on patients admissions index distribution
depth_map = pd.Series(sampled_depths, index=unique_controls)

# Sort the control admissions by SUBJECT_ID and ADMITTIME to ensure alignment prior to calculating cumulative count for event_index assignment
CONTROL_ADMISSIONS_MERGED = CONTROL_ADMISSIONS_MERGED.sort_values(['SUBJECT_ID', 'ADMITTIME'], ascending=[True, False])

# Compute the number of admissions for each control patient and assign temporary rank to each admission
CONTROL_ADMISSIONS_MERGED['temp_rank'] = CONTROL_ADMISSIONS_MERGED.groupby('SUBJECT_ID').cumcount()

# Only keep the most recent n admissions for each control patient, where n is the number of sampled admissions for a diseased patient with the same distribution.
CONTROL_ADMISSIONS_MERGED = CONTROL_ADMISSIONS_MERGED[
    CONTROL_ADMISSIONS_MERGED['temp_rank'] < CONTROL_ADMISSIONS_MERGED['SUBJECT_ID'].map(depth_map)
].copy()

# Compute this hypothetical "admissions index" for controls, with same distribution as patients, assigning to event_index column to ensure a homogenous distribution
CONTROL_ADMISSIONS_MERGED['event_index'] = (CONTROL_ADMISSIONS_MERGED['temp_rank'] + 1) * -1
CONTROL_ADMISSIONS_MERGED.drop(columns=['temp_rank'], inplace=True)

Diagnoses appearing < 100 times: 6103
Procedures appearing < 100 times: 1749
Final filtered diagnoses count: 881
Final filtered procedures count: 260


In [51]:
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import MultiLabelBinarizer

# This function extracts the length of stay for patients, which will be one of the supporting features for the model. It also drops noisy columns that could cause data leakage, such as HADM_ID, ADMITTIME, DISCHTIME, DEATHTIME, EDREGTIME, EDOUTTIME, HOSPITAL_EXPIRE_FLAG, and the text-based diagnosis label since we have the ICD-9 diagnosis codes. This function is applied to both the patients and control dataframes to ensure they have the same structure and features for modeling.
def preprocess_clinical_data(df):
    df = df.copy()
    
    # Calculate length of stay in days
    df['ADMITTIME'] = pd.to_datetime(df['ADMITTIME'])
    df['DISCHTIME'] = pd.to_datetime(df['DISCHTIME'])
    df['LOS_DAYS'] = round((df['DISCHTIME'] - df['ADMITTIME']).dt.total_seconds() / 86400)
    
    # Define Leaky/Identifier columns to drop
    cols_to_drop = [
        'HADM_ID', 'ADMITTIME', 'DISCHTIME', 
        'DEATHTIME', 'EDREGTIME', 'EDOUTTIME', 'HOSPITAL_EXPIRE_FLAG',
        'DIAGNOSIS (LABEL)' # Text label is redundant to ICD9 codes
    ]
    
    # Drop columns if they exist in the dataframe
    existing_drops = [c for c in cols_to_drop if c in df.columns]
    df_clean = df.drop(columns=existing_drops)
    
    return df_clean

# Executing on both, cleaning data
PATIENT_CLEAN = preprocess_clinical_data(PATIENT_ADMISSIONS_MERGED)
CONTROL_CLEAN = preprocess_clinical_data(CONTROL_ADMISSIONS_MERGED)

In [ ]:
def prepare_patient_level_features(PATIENT_CLEAN, CONTROL_CLEAN):
    # Combine everything into one DF to ensure columns align across all patients
    df = pd.concat([PATIENT_CLEAN, CONTROL_CLEAN], axis=0)
    

    # This function extracts the codes from the condensed ICD-9 codes columns per admission, and then returns a list of column labels accounting for both the event index of the row and the ICD9 codes.
    # This allows us to retain temporal structure by assigning codes from each admission to one of the "admission slots" based on event index, which is critical for the model to learn temporal patterns
    def prefix_codes(row, col_name):
        codes = row[col_name]
        if not isinstance(codes, list): return []
        # event_index is -1, -2, etc. We use int() to keep it clean.
        return [f"T{int(row['event_index'])} {code}" for code in codes]

    # This code generates new lists for each admission's diagnoses and procedures, expanding the dataframe horizontally and preparing for one hot encoding while retaining temporal structure. Each code is prefixed with the time slot (T-1, T-2, etc.) to allow the model to learn temporal patterns in the sequence of diagnoses and procedures.
    df['diag_temporal'] = df.apply(lambda x: prefix_codes(x, 'DIAGNOSIS (ICD_9)'), axis=1)
    df['proc_temporal'] = df.apply(lambda x: prefix_codes(x, 'PROCEDURE TYPE'), axis=1)

    # Now we concatenate all the existing diagnoses and procedures temporal codes into single lists per patient, we capture the max length of stay value to capture only the worst severity stay length for each patient to prevent noise
    # Demographic factors remain constant so we will just extract the first value
    patient_df = df.groupby('SUBJECT_ID').agg({
        'diag_temporal': 'sum',
        'proc_temporal': 'sum',
        'LOS_DAYS': 'max',
        'ADMISSION_TYPE': 'first', # Demographics stay the same across visits
        'INSURANCE': 'first',
        'ETHNICITY': 'first'
    })

    #Impute any NAs for datetime computed LOS_DAYS with 0s so that model can use this feature without issues. Get dummies is called to one hot encode demographic categories
    X_num = patient_df[['LOS_DAYS']].fillna(0)
    X_cat = pd.get_dummies(patient_df[['ADMISSION_TYPE', 'INSURANCE','ETHNICITY']], dummy_na=True)

    # Manually drop the Newborn column if it exists since newborn status was found to be a top feature. If there are many newborn cases we don't want the model to pick "positive" or "negative" based on this status since newborns are less likely to get bacterial endocarditis
    if 'ADMISSION_TYPE_NEWBORN' in X_cat.columns:
        X_cat = X_cat.drop(columns=['ADMISSION_TYPE_NEWBORN'])    
    return X_num, X_cat, patient_df['diag_temporal'], patient_df['proc_temporal'], patient_df.index, df

# Extract numerical, and categorical features, along with temporal diagnosis and procedure lists for each patient 
num_f, cat_f, diag_list, proc_list, patient_ids, full_df = prepare_patient_level_features(PATIENT_CLEAN, CONTROL_CLEAN)

# Prepare one hot encoding for diagnosis and procedures using MultiLabelBinearizer
mlb_diag = MultiLabelBinarizer(sparse_output=True).fit(diag_list)
mlb_proc = MultiLabelBinarizer(sparse_output=True).fit(proc_list)

# Stack one hot encoded diagnosis and procedure features with numerical and categorical features into a final sparse matrix for modeling.
X_sparse = hstack([
    mlb_diag.transform(diag_list),
    mlb_proc.transform(proc_list),
    csr_matrix(num_f.astype(float)),
    csr_matrix(cat_f.astype(float))
])

# Create target y based on whether a patient is in diseased or controls DF
y = np.array([1 if pid in DISEASE_SUBJECT_ID else 0 for pid in patient_ids])

print(f"New Patient-Level X Shape: {X_sparse.shape}")
print(f"Total Patients: {len(y)} | Diseased Patients: {sum(y)}")

New Patient-Level X Shape: (146, 1470)
Total Patients: 146 | AD Patients: 73


In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, roc_auc_score

# 1. Setup 5-Fold CV: 5 splits, shuffle to ensure random distribution of patients across folds, and set random state for reproducibility
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_metrics = []
importances = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_sparse, y)):
    X_tr, X_te = X_sparse[train_idx], X_sparse[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]
    
    ratio = (y_tr == 0).sum() / (y_tr == 1).sum()
    
    model = XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=3,
        scale_pos_weight=ratio,
        random_state=42,
        eval_metric='logloss'
    )
    
    model.fit(X_tr, y_tr)
    
    # Probabilities and Predictions (Default 0.5 threshold)
    probs = model.predict_proba(X_te)[:, 1]
    preds = model.predict(X_te)
    
    # Calculate Metrics
    auc = roc_auc_score(y_te, probs)
    tn, fp, fn, tp = confusion_matrix(y_te, preds).ravel()
    
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    fold_metrics.append({'Fold': fold+1, 'AUC': auc, 'Sens': sens, 'Spec': spec})
    importances.append(model.feature_importances_)
    
    print(f"Fold {fold+1} | AUC: {auc:.3f} | Sens: {sens:.1%} | Spec: {spec:.1%}")

# Final Summary
summary_df = pd.DataFrame(fold_metrics)
print("\n--- FINAL MEAN METRICS ---")
print(summary_df[['AUC', 'Sens', 'Spec']].mean())

# Feature Importance
avg_imp = np.mean(importances, axis=0)
feat_names = mlb_diag.classes_.tolist() + mlb_proc.classes_.tolist() + ['LOS_DAYS'] + cat_f.columns.tolist()
feat_df = pd.DataFrame({'Feature': feat_names, 'Importance': avg_imp}).sort_values('Importance', ascending=False)

print("\n--- TOP 300 PREDICTORS ---")
print(feat_df.head(300))

Fold 1 | AUC: 0.796 | Sens: 66.7% | Spec: 86.7%
Fold 2 | AUC: 0.819 | Sens: 80.0% | Spec: 78.6%
Fold 3 | AUC: 0.619 | Sens: 66.7% | Spec: 57.1%
Fold 4 | AUC: 0.633 | Sens: 57.1% | Spec: 53.3%
Fold 5 | AUC: 0.610 | Sens: 42.9% | Spec: 60.0%

--- FINAL MEAN METRICS ---
AUC     0.695302
Sens    0.626667
Spec    0.671429
dtype: float64

--- TOP 20 PREDICTORS ---
        Feature  Importance
449    T-1 V433    0.045984
1099   T-1 3521    0.044780
1255   T-2 3995    0.043552
1293   T-2 9904    0.037324
546   T-2 40391    0.035164
...         ...         ...
277    T-1 5920    0.000000
276     T-1 591    0.000000
275   T-1 59010    0.000000
274    T-1 5859    0.000000
273    T-1 5856    0.000000

[300 rows x 2 columns]


In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score
import numpy as np
import pandas as pd

# Prune Features: Use only the Top 300 from previous run
top_300_indices = feat_df.head(300).index.tolist()
# Map the indices back to the sparse matrix columns
X_reduced = X_sparse[:, top_300_indices]

# Setup 5-Fold CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
reduced_results = []

print(f"Running model on {X_reduced.shape[1]} selected features...")

for fold, (train_idx, test_idx) in enumerate(skf.split(X_reduced, y)):
    X_tr, X_te = X_reduced[train_idx], X_reduced[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]
    
    ratio = (y_tr == 0).sum() / (y_tr == 1).sum()
    
    # 3. High Regularization to prevent overfitting to 27 patients
    model = XGBClassifier(
        n_estimators=200,      # Fewer trees to prevent "memorizing" patients
        learning_rate=0.02, 
        max_depth=2,           # Shallow trees are more robust
        reg_lambda=15,         # High L2 penalty
        scale_pos_weight=ratio,
        eval_metric='logloss',
        random_state=42
    )

    model.fit(X_tr, y_tr)
    
    probs = model.predict_proba(X_te)[:, 1]
    # Lower threshold slightly to 0.4 to capture the rare positive class
    preds = (probs >= 0.4).astype(int)
    
    auc = roc_auc_score(y_te, probs)
    tn, fp, fn, tp = confusion_matrix(y_te, preds).ravel()
    
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    reduced_results.append({'Fold': fold+1, 'AUC': auc, 'Sens': sens, 'Spec': spec})
    print(f"Fold {fold+1} | AUC: {auc:.3f} | Sens: {sens:.1%} | Spec: {spec:.1%}")

print("\n--- REDUCED FEATURE MEAN METRICS ---")
print(pd.DataFrame(reduced_results)[['AUC', 'Sens', 'Spec']].mean())

Running model on 300 selected features...
Fold 1 | AUC: 0.858 | Sens: 86.7% | Spec: 53.3%
Fold 2 | AUC: 0.657 | Sens: 80.0% | Spec: 50.0%
Fold 3 | AUC: 0.705 | Sens: 80.0% | Spec: 42.9%
Fold 4 | AUC: 0.414 | Sens: 57.1% | Spec: 33.3%
Fold 5 | AUC: 0.736 | Sens: 92.9% | Spec: 40.0%

--- REDUCED FEATURE MEAN METRICS ---
AUC     0.673937
Sens    0.793333
Spec    0.439048
dtype: float64
